# [AINIT Project 3] Solving the Capacitated Vehicle Routing Problem with Reinforcement Learning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title ⚙️ Setup: download project data and install packages { display-mode: "form" }
import os, subprocess, sys, shutil
# Always start fresh to avoid stale state
if os.path.exists("/content/project"):
    shutil.rmtree("/content/project")
PROJECT_DIR = "/content/project/Project3" # where the project is cloned to in the colab VM
# 🎯 TODO: Change the following to your own path in Google Drive where you persist data
DRIVE_DIR = "/content/drive/MyDrive/project/Project3" 
os.makedirs(DRIVE_DIR, exist_ok=True) # Create the folder if it doesn't exist

repo_url = f"https://github.com/eth-ainit-fs26/project.git"
branch = "week3"
subprocess.run(["git", "clone", "--branch", branch, "--single-branch", repo_url, "/content/project"], check=True)
print("✅ Repo successfully clone to /content/project")

os.chdir(PROJECT_DIR)
!uv pip install --system --break-system-packages -r requirements.txt
print("✅ Package installation complete — working directory:", os.getcwd())

# Part 1. Introduction

## 1.1 Why CVRP?

Logistics and supply chain are the engine of global commerce, and at the core of efficiency lies the Capacitated Vehicle Routing Problem (CVRP). Simply put, it is the challenge of finding the shortest and most efficient set of routes for a fleet of vehicles to serve many different customer locations while ensuring that no single vehicle's carrying capacity is ever exceeded. By minimizing total time and distance traveled, effective CVRP solutions directly cuts major operational expenses like fuel, labor wages, and vehicle maintenance.


CVRP is formally classified as an NP-hard problem, meaning that finding exact solutions becomes prohibitively slow as the size of the network grows. Traditionally, this problem has been approached using *heuristics* (rule-of-thumb methods like Nearest Neighbor) and *metaheuristics* (higher-level search strategies like local search and Simulated Annealing) which are designed to quickly find good but not necessarily optimal solutions.

In recent years, a new line of research focuses on training AI models to solve these hard problems. RL is particularly suitable for this because the return is easily calculated (e.g. the negative travel cost), and we can let the RL model self-play on a large number of instances to improve itself. Similar concepts have led to great success like AlphaZero for chess/shogi/go, AlphaFold for protein structure prediction, and AlphaProof and AlphaGeometry for mathematical problem solving.

One major challenge in building a performant model is to represent our data in a concise and informative way so that the model can make good decisions, and in modern AI approach this means coming up with vector representations in a smart way. For CVRP, the vectors should capture the relative locations of all the customers as well as their demands. This should ring a bell: we did something similar in Weekend 1 for texts using the attention mechanism! The attention mechanism is the backbone of modern LLMs; as the name suggests, it allows the model to "pay attention to" parts of your data which matter. In this project, instead of generating text, we will train a model which iteratively suggests which customer to visit next based on the history. Specifically, we will implement the POMO algorithm from the paper [POMO: Policy Optimization with Multiple Optima for Reinforcement Learning](https://arxiv.org/abs/2010.16011), which built upon [this pioneering work](https://arxiv.org/abs/1803.08475) from 2019.


This project quite dense and theoretical, but we hope that you find this project rewarding by focusing more on developing conceptual understanding rather than getting lost in the implementation. Even though this notebook looks very long, please don't panic as we try to provide as detailed explanations as possible. All the parts we ask you to implement are marked with 🎯 TODO.


Without further ado, buckle up and let's begin!

## 1.2 What is CVRP exactly?

In [ ]:
from IPython.display import Image, display
display(Image('cvrp_demo.png', width=600))

<!-- <img src="cvrp_demo.svg" alt="CVRP Instance" style="display: block; margin: 0 auto; width:800px;"/> -->
To fully understand what we are working with, it is important to first take a step back and define the problem more formally. The image above shows a CVRP instance with 16 customers along with a solution which uses 4 vehicles (corresponding to the 4 colored routes).

### Instance (input data)
An ***instance*** is defined by $N$ customers (labeled as $1,\ldots, N$) and a depot (labeled as $0$); in the image above, the labels are the numbers inside each node
- The customers and the depot are referred to as **nodes**, and $N$ is called the **problem size**.
- Each customer node $i$ has some demand $d_i$ (from $1$ to `MAX_DEMAND`) for delivery (the number next to each node in the image). You may think of this as the weight or size of the customer's order. The depot node has no demand, or for convenience we say that it has demand $0$.
- We assume access to a large enough fleet of vehicles to serve all customers, each of them with capacity `VEHICLE_CAPACITY`>0.



### Solution
A ***solution*** to an instance is a collection of **routes** serving all customer nodes exactly once. A route is the journey taken by one vehicle and satisfies the following:
- A route must start at the depot, end at the depot and serve at least one customer
- The sum of customer demands along that route must not exceed `VEHICLE_CAPACITY`

In our example, the solution (up to permutation of vehicles) is
\begin{align*}
[ \quad& \\
    &[0, 4, 3, 1, 7, 0], \\
    &[0, 8, 2, 6, 5, 0], \\
    &[0, 14, 16, 10, 9, 0], \\
    &[0, 12, 11, 15, 13, 0] \\
]\quad &
\end{align*}

Another way to view a solution is to concatenate all the routes together into a single ***tour*** $\tau$, as if we are using only one vehicle which can reload at the depot for the next route. In our example, this becomes

$$
\tau = [0, 4, 3, 1, 7, 0, 8, 2, 6, 5, 0, 14, 16, 10, 9, 0, 12, 11, 15, 13, 0].
$$

We adopt this more convenient viewpoint for RL to avoid keeping track of multiple vehicles explicitly.

### Cost

The quality of a solution tour $\tau$ is measured by some cost measure. This could be a combination of distance, time, fixed operation cost per vehicle, penalties for not visitng a customer, etc. In this project, we choose to work with an estimated fuel consumption to complete the tour $\tau$. Specifically, the fuel consumption in L to complete $\tau$ is given by

$$
\text{cost}(\tau) := \alpha*\text{distance (m)} + \beta * \text{time (s), where} $$
$$\alpha= 0.0001 \text{ L/m} \quad \text{and} \quad \beta=0.001 \text{ L/s}$$

Solving CVRP means finding a feasible (i.e. valid) solution with the lowest cost possible.

The parameters defining CVRP instances can be found in `source.parameters.py`. We take these values as assumptions and do not change them in this project.

## 1.3 Working with the real road network

The picture above is of course highly simplified and unrealistic: Our depot and customers should be connected via actual road networks instead! To work in a more realistic setting, we need to address the following complications:

1. **Distance Computation**: Instead of the straight-line distance, we need to find the cheapest path from point A to point B via some routing algorithm
2. **Travel Time Estimation**: To compute the travel cost, we need to estimate the travel speed along each road


We will work with OpenStreetMap data in Zurich. OpenStreetMap (OSM) is a free, open-source, and collaborative map of the world. Instead of being owned by a massive corporation like Google or Apple, it is built and continuously updated by a global community of volunteers who contribute data about roads, buildings, trails, cafes, and much more.

In order not to distract you from RL, we have handled all the complexities behind the scene. We invite those interested to check the script `source/map_utils.py` for details.

# Part 2: Prepare road network

## 2.1 Preprocessing

As the first step, we need to download the data from OpenStreetMap and do some preprocessing to perform vehicle routing. Below, you first run the `prepare_zurich_environment` which does the following:
1. Download the driving network in Zurich and take its largest strongly connected component to ensure reachability. This is then projected into the Universal Transverse Mercator (UTM) coordinate system and saved in the variable `G_utm`. Edges and nodes of the graph corresponds to (directed) drivable roads and their endpoints, and edges are marked with their lengths and speed limits.
2. In order to better estimate the actual travel time, congestion factors (depending on the road type) are multiplied to the speed limits as the average travel speed.
3. The cost of traversing through each edge is computed and added to the graph.
4. Since routing will be done on nodes of `G_utm` (which is not that large), we can avoid re-computations by computing the cheapest path cost between each pair of points and save it in the `apsp` matrix: `apsp[i][j]` would be the cheapest cost from node `i` to `j`. Notice that this matrix is not symmetric.
5. Each OSM node is represented by an ID. To simpify things, we map those IDs to `range(0, num_nodes_in_G_utm)` and store this correspondence in the variable `node_map`.
<!-- 6. For numerical stability in RL training, we add normalized `x` and `y` utm coordinates to each node in `G_utm`. -->

To see what the preprocessing actually produces, consider the following toy example with just 4 intersections: a depot $D$ and three customers $A,B,C$. The cell below starts from their geocoordinates (lat/lon) and displays the processed data as follows:

- $\fbox{Node table}$ This table displays the nodes' latitude/longitude in degrees, UTM easting/northing in meters and the mapping of mock OSM IDs to indices.
  
- $\fbox{Left panel}$ This panel shows the road network after Steps 1–3. Each arrow is a directed road segment annotated with the mock edge cost `length @ speed = travel time`. The street $A\to B$ is **one-way** (red arrow). Nodes are plotted at their normalized coordinates, with the original lat/lon and UTM values shown next to each.
  
- $\fbox{Right panel}$ This panel visualizes the precomputed `apsp` matrix (step 4). Every entry is the cheapest travel time between a pair of nodes. The highlighted cells show that the matrix is **asymmetric**: going $A\to B$ takes 72 s on the direct one-way street, but coming back $B\to A$ is forbidden, so the cheapest return path is the detour $B\to C \to D\to A \approx 206 \text{ s}$.

Note that for simplicity we displayed the cost as the the travel time, but in training we will use the cost defined in Section 1.2 which combines distance and time.

In [ ]:
# @title Toy example { display-mode: "form" }
from source import map_utils # Map-related functions
map_utils.visualize_toy_example()

In [ ]:
# @title Import Libraries  { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from seaborn import heatmap
import pickle
from IPython.display import HTML
from source.cvrp_generator import CVRPGenerator # Instance generator
import source.utils_or_tools as Baseline # Baseline solver (OR-Tools)
from source.utilities import visualize_solver_performance # Solver evaluation visualization

In [ ]:
# @title Download Zurich Environment { display-mode: "form" }
# WARNING: This file is large. Run this cell only once to save to a file
# and read from the saved file in the future. This also ensures reproducibility
# since the map data can change over time.
G_utm, apsp, node_map = map_utils.prepare_zurich_environment()
persist = True # whether you want to persist the environment to your drive
which_dir = DRIVE_DIR if persist else PROJECT_DIR
with open(f"{which_dir}/zurich_environment.pkl", "wb") as f:
    pickle.dump((G_utm, apsp, node_map), f)
    print(f"✅ Zurich driving network saved to {which_dir}/zurich_environment.pkl")

In [ ]:
# @title Load Downloaded Zurich Environment { display-mode: "form" }
# Once you have the file zurich_environment.pkl saved, comment the cell above
# and uncomment this cell to load the environment from the file in the future.
which_dir = DRIVE_DIR # DRIVE_DIR or PROJECT_DIR
with open(f"{which_dir}/zurich_environment.pkl", "rb") as f:
    G_utm, apsp, node_map = pickle.load(f)
    print(f"Successfully loaded Zurich network from {which_dir}/zurich_environment.pkl")
# number of nodes and edges of G_utm
print("G_utm has", G_utm.number_of_nodes(), "nodes and", G_utm.number_of_edges(), "edges.")

## 2.2 Data & Baseline Solver

In this section we explore how random CVRP instances can be generated, solved and visualized.

Why do we need to generate instances? This is one of the key differences between this reinforcement learning setting and a standard machine learning setup. In most supervised learning problems, we start from a fixed dataset: each training example is already available and usually comes with a correct label or target value. The model learns by comparing its prediction with this known answer.

Here, however, we do not have a dataset of pre-solved CVRP instances. A training example is not a single labelled observation, but an entire routing problem: a set of customer locations on the Zurich map, their demands, and the travel costs between them. For each generated instance, the actor has to construct a solution by itself. The quality of that solution, measured through the total route cost, is then used as the learning signal.

Therefore, instances are generated on the fly during training so that the actor can practice on many different CVRP problems of the same type. If we trained it only on one fixed configuration of customers, the model could simply memorize that specific solution. By generating a continuous stream of new instances, we force the actor to learn a general routing strategy that can generalize to unseen customer configurations.

In conclusion, our goal is to train an actor which can solve any given vehicle routing instances. This means the training data should be randomly generated and diverse enough to ensure that the model can generalize to unseen tasks. The core sampling logic is defined inside the class `CVRPGenerator`, which we now explain.

The class method `CVRPGenerator.sample_instance(n)` randomly generates a CVRP instance with $n$ nodes (including depot) as follows:
1. Randomly sample $n$ edge IDs with replacement from the graph `G_utm` and store them in `edge_indices`.
2. Randomly sample $n$ numbers from $0$ to $1$ and store them in $t$.
3. The sampled points are then computed as the linear interpolations between the source and target of the sampled edges. More precisely, an edge $u\to v$ together with a fraction $t$ represents the point $p=u+t(v-u)$.
4. The associated pairwise travel cost matrix is computed and returns as `cost_matrix`.
5. Randomly sample $n-1$ integer-valued demands from $1$ to `MAX_DEMAND` (since the depot always has demand $0$).


To sample multiple instances at once, you may use the method `CVRPGenerator.sample_batch(batch_size, num_nodes)`.

We will use [Google OR-tools](https://developers.google.com/optimization) as the baseline solver for this project, a free and open-source software suite developed by Google for solving various types of optimization problems. This solver uses *heuristics* to find an initial feasible solution, and then employs *metaheuristics* which make local changes to the current solution to arrive a better solution.

$\fbox{Initialize the generator}$

Create the `CVRPGenerator` on the Zurich environment (`G_utm`, `apsp`, `node_map`). You may pass in a random number generator for reproducibility.

In [ ]:
# @title Initialize the CVRP generator { display-mode: "form" }
# Initialize a generator
rng = np.random.default_rng(seed=10)  # For reproducibility
generator = CVRPGenerator(G_utm, apsp, node_map, rng=rng)

$\fbox{Sample one instance and inspect its costs}$

Draw a random instance with 10 customers + depot and plot its (asymmetric) pairwise cost matrix.

In [ ]:
# @title Sample an instance & plot its cost matrix { display-mode: "form" }
# First, let's look at a random sample
num_customers = 10
num_locations = num_customers + 1  # including the depot
edge_indices, t, demands, cost_matrix = generator.sample_instance(num_locations=num_locations)
# Visualize the pairwise cost matrix
fig, ax = plt.subplots(figsize=(8,6))
heatmap(cost_matrix, ax=ax, annot=True, fmt=".2f", cmap="viridis")
ax.set_title(r'Estimated Fuel Consumption (L) Between Locations', fontsize=14)
ax.set_xlabel('Destination'), ax.set_ylabel('Source')
plt.show()


$\fbox{Pack node information into a DataFrame}$

Solvers and visualization functions take in node information as a DataFrame (one row per location); `.head(k)` shows the first $k$ rows.

In [ ]:
# @title Convert the instance to a DataFrame { display-mode: "form" }
# Convert to DataFrame for easier handling in visualization and solvers
cvrp_instance = map_utils.convert_node_info_to_dataframe(edge_indices, t, demands)
cvrp_instance.head(5)

$\fbox{View the instance on the map}$

With `cvrp_solution=None` only depot and customers are drawn, no routes yet.
You may control what information to display with the toggle in the top-right corner.

In [ ]:
# @title Map of depot & customers (no routes) { display-mode: "form" }
cvrp_solution = None
m0 = map_utils.visualize_cvrp_solution(G_utm, generator, cvrp_instance, cost_matrix,
                                       cvrp_solution, fname="cvrp_stops.html")
display(m0) # comment out this line to disable rendering

$\fbox{Hand-craft a solution}$

As a start, we eye-ball a solution; feasible, but probably far from optimal.

In [ ]:
# @title Hand-crafted solution on the map { display-mode: "form" }
# Eye-ball a solution and draw it on the map
cvrp_solution = [
    [0, 4, 8, 10, 5, 3, 6, 9, 0],
    [0, 2, 7, 1, 0]
]
m1 = map_utils.visualize_cvrp_solution(G_utm, generator, cvrp_instance, cost_matrix,
                                       cvrp_solution, fname="cvrp_solution.html")
display(m1) # comment out this line to disable rendering

$\fbox{Solve with OR-Tools}$

The baseline solver computes routes for the same instance, to be compared against the hand-made solution (and later against the RL model).

In [ ]:
# @title Solve with OR-Tools & map the solution { display-mode: "form" }
# Solve with OR-Tools and visualize the solution
or_solution = Baseline.solve_with_ortools(cost_matrix, demands)
m2 = map_utils.visualize_cvrp_solution(G_utm, generator, cvrp_instance, cost_matrix,
                                       or_solution['routes'], fname="cvrp_solution_ortools.html")
display(m2) # comment out this line to disable rendering

$\fbox{Compare the two solutions}$

You can create a dual-panel map comparing two solutions as follows:

In [ ]:
# @title Side-by-side comparison map { display-mode: "form" }
# Compare the two solutions side by side
map_utils.visualize_two_cvrp_solutions(
    G_utm, generator, cvrp_instance, cost_matrix,
    cvrp_solution_1=cvrp_solution, cvrp_solution_2=or_solution['routes'],
    panel_title_1="Hand-Crafted Solution", panel_title_2="OR-Tools Solution",
    fname="cvrp_comparison.html"
)

## 2.3 Evaluation of Baseline solver

To evaluate our baseline solver, we will sample a number of instances for each customer size and record the solution cost as well as the solving time.

In [ ]:
# @title Solve instance of each size with OR-Tools and visualize the performance { display-mode: "form" }
or_eval_size, or_eval_seed = 30, 99
or_cost, or_time = Baseline.evaluate_baseline_solver(
    n_instances_per_size=or_eval_size, generator=generator, seed=or_eval_seed)
fig_or = visualize_solver_performance(cost_baseline=or_cost, time_baseline=or_time, baseline_only=True,
    n_instances_per_size=or_eval_size, eval_seed=or_eval_seed, figsize=(12,4)
)

You may argue that this works fine, but the issues with OR-Tools is that both the computation time and optimality gap grows significantly as the instance size grows.

In this project, we explore how RL can be used to solve this hard combinatorial optimization problem. Our RL model will be able to solve multiple instances of the same size in parallel, thus resulting in massive speedup. In the POMO paper (Table 3), the authors experimented with 10,000 random CVRP100 instances (i.e. 100 customers): OR-Tools gave substantially worse solutions than POMO after 1 hour, whereas POMO returned near-optimal solutions in just 2 minutes! In a nutshell, RL when done right can give you better CVRP solutions than OR tools while being faster.


Remark: A baseline solver refers to an existing method that provides a minimum expected performance for a given task. It serves as a reference point against which the performance of new models is measured. This is not to be confused with the baseline in REINFORCE with Baseline which refers to a quantity subtracted from the return used in the policy gradient calculation to achieve variance reduction and better "credit assignment".

# Part 3: Model CVRP as an RL problem


The next step is to convert CVRP into an RL problem. This means we need to define the RL environment by specifying the states, the actions, the state transition logic and the rewards. This conversion is not always straightforward, but one can often come up with a reasonable model by mimicking what a human would do.

We would like our RL model to generate a solution tour by choosing one node after another. Thus it is natural to define a state as a (partial) solution tour of an input instance and an action as the next node to visit given the tour history. Upon taking an action, the current (partial) tour is extended by appending the visited node to the current node list. Only when a solution (terminal state $\tau$) is reached will the agent be given a reward of $-\text{cost}(\tau)$. In RL we call this an *episodic* environment: the agent interacts with an environment which reaches an end after some steps. Since the reward is only given at the end of the episode, the return $R(\tau)$ of this episode is equal to that reward, namely
$$R(\tau) := -\text{cost}(\tau).
$$

Recall that a policy is a mapping from state to a probability distribution over the available actions. You may think of the RL agent as a policy network (i.e. a neural network that defines a policy) or an ***actor*** in RL jargon, which receives the current state as an input and proposes the next action to take based on a probability distribution.

## 3.1 Node Input Features (and Why)

Choosing the right information to feed into a model (feature engineering) is crucial to its success.
The information we choose need to give the model the right spatial and structural context.
For each node, we compute the following `NODE_FEATURE_DIM=14` features (where "distance" means cost):

<center>

|         Feature(s)        | What the model learns |
|------------------------|--------------|
| normalized demand      | Can I physically fit this node into my current truck? |
|  min in- and out-distance       |  Is there an obvious "next step" right next door?  |
|  mean and std of in- and out-distance   |  Where does this node sit in the grand scheme of the entire map? |
| Percentiles (p10, p50, p90) of in- and out-distance |  What does the surrounding neighborhood look like? Is it dense or sparse? |
</center>



Notice that we are not using the coordinates of the nodes, because proximity in coordinates does not always mean short travel cost. Instead, we will somehow feed the pairwise travel cost matrix directly into our model.

## 3.2 Transformer Encoder

There is one major challenge we need to address before going any further. The backbone of our RL actor will be a neural network (NN), which takes in the state as input and produces a prboability distribution over the next actions. As we know, NNs must take ***fixed-sized, continuous*** input vectors, but our RL state is something ***discrete/combinatorial and of varying length*** (list of nodes). How do we patch this gap in a smart way?

To solve this, we employ a ***Transformer Encoder*** to map the discrete graph structure into a continuous latent space. At a high level, a Transformer Encoder is a neural network architecture designed to look at a collection of data points (like nodes on a map) and understand each point in the context of everything else around it. Instead of reading data sequentially from left to right, a Transformer encoder processes the entire input all at once. Its primary goal is to turn a list of isolated data points into a list of contextualized representations (or embeddings). You may think of this as learning a representation of the RL environment.

Understanding the attention mechanism is beyong the scope of this project. Here is a short explanation for your intuition: At its core, the attention mechanism asks

<center>
<span style="color: red; font-weight: bold; font-style: italic; font-size: 20px;">
Based on my current feature state (Query) and your feature state (Key), how much attention should I pay to you?
</span>
</center>

The attention node $i$ pays to node $j$ is computed as

$$\text{Score}_{ij} = \frac{\langle Q_i, K_j\rangle}{\sqrt{d_k}} + \text{MLP}(\text{Cost}_{ij}),$$

where $Q_i$ is the query, $K_j$ is the key, $\langle\cdot,\cdot\rangle$ is the dot product, $d_k$ is the key dimension `KEY_DIM`, $\text{Cost}_{ij}$ is the travel cost from $i$ to $j$, and MLP is a small NN.


In the end, the encoder produces contextualized node embeddings $\mathbf{h}_0,\mathbf{h}_1,\ldots,\mathbf{h}_N \in\mathbb{R}^\text{EMBEDDING\_DIM}$, where each embedding contains information not only about the corresponding location itself, but also about the entire instance. We also compute a graph embedding by averaging all node embeddings:

$$\bar{\mathbf{h}} = \frac{1}{N+1}
\left(\mathbf{h}_0+\mathbf{h}_1+\cdots + \mathbf{h}_N\right)$$

This graph embedding acts as a global summary of the whole CVRP instance. It gives the decoder a compact representation of the overall delivery problem, including the spatial layout, the graph structure, and the demand distribution.

## 3.3 Decoder

The decoder is the main part of our RL logic which constructs the vehicle tour step by step.
Given a sequence of nodes already visited, it "decodes" the input sequence and pick the next node to visit.
Let's say our vehicle has already visited the tour $\tau = (\tau_0,\tau_1,\ldots,\tau_{t})$ with remaining capacity $\text{rem}(\tau)$. The decoder builds a context vector by concatenating three pieces of information:

$$
\widetilde{\mathbf{h}} = \left[ \bar{\mathbf{h}}, \mathbf{h}_{\tau[t]}, \text{rem}(\tau) \right] \in\mathbb{R}^{2\times\text{EMBEDDING\_DIM} +1}
$$
This context vector tells the decoder:

1. what the overall problem looks like, through the graph embedding $\overline{\mathbf{h}}$;
2. where the vehicle currently is, through the embedding of the last visited node $\mathbf{h}_{\tau[t]}$;
3. how much capacity is still available, through $\text{rem}(\tau)$.

Using this context vector as a query, the decoder attends over all node embeddings $\mathbf{h}_0,\mathbf{h}_1,\ldots,\mathbf{h}_N$ using the attention mechanism, scores every possible next node and produces a probability distribution over the $N+1$ locations.

However, not all nodes are feasible at every step. Customers that have already been visited cannot be selected again, and customers whose demand exceeds the remaining vehicle capacity cannot be visited before returning to the depot. These hard constraints are enforced through a mask. The mask assigns $-\infty$ to infeasible nodes before the softmax operation. As a consequence, infeasible nodes receive probability zero.

The decoder thus outputs a vector with $N+1$ entries, each representing the probability of choosing a specific node as the next node in the tour. After a node is selected, the environment updates as follows:

- if the selected node is a customer, that customer is marked as visited and its demand is subtracted from the remaining vehicle capacity;
- if the selected node is the depot, the vehicle starts a new route and its capacity is refilled.

The decoder then receives the updated tour as input and produces a probability distribution over the next node accordingly.
This process is repeated until all customers have been served. Once the tour is complete, its tour cost will serve as the reward signal for training.


The neural networks for the encoder and the decoder has been implemented for you in `source.grouped_actor.py`. Below you will only need to implement parts of the decoder related to RL.

## 3.4 Input data format for RL

We will use PyTorch to implement our RL model, and our (random) datasets will be generated using the class `DATALOADER` which extends  `torch.utils.data.DataLoader`. Since we have already implemented the logic of sampling random CVRP instances in `CVRPGenerator`, we will wrap this logic inside `DATALOADER`.

The dataloader packs together the following information:
1. Node demands: unnormalized, integer-valued demands for each node
2. Cost matrix: pairwise travel cost matrix of the instance.

We handle the tensor data in batches. If `batch_size` is the batch size and `problem_size` (or `p_size`) is the number of customers, then the tensors above have the following types and shapes:
1. Node demands: `torch.LongTensor` of size `(batch_size, p_size+1, 1) `
2. Cost matrix: `torch.FloatTensor` of size `(batch_size, p_size+1, p_size+1)`

You may use a for loop to iterate through the data loader and get batches of data. Each batch is allowed to have a different `p_size`, which you can control using the arguments `problem_sizes_mean` and `problem_sizes_std`. The two arguments specify the mean and standard deviation of the Gaussian distribution from which `p_size` will be sampled (before rounding to an integer between `MIN_NUM_CUSTOMERS` and `MAX_NUM_CUSTOMERS`). They can be either numbers or lists of numbers of length `batch_size`. This is explained in the `data_loader_demo` function below.



In [ ]:
# @title Import Libraries and Set up Device { display-mode: "form" }
import torch # PyTorch library for deep learning
from source import cvrp
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu' # Use this line to run on Colab/CUDA-enabled machines
# DEVICE = 'mps' if torch.mps.is_available() else 'cpu' # Use this line to run on Apple Silicon
DATALOADER = cvrp.DATALOADER # shorthand for data loader class in cvrp module
print("🖥️ Using device:", DEVICE)

In [ ]:
# @title Data Loader Demo { display-mode: "form" }
def data_loader_demo(dataloader, description="Data Loader Info"):
    print(f"{description:=^90}")
    print(f"Total number of data points: {len(dataloader.dataset)}")
    for i, (dems, cost_mat) in enumerate(dataloader):
        feats = cvrp.compute_batched_features(generator, cost_mat, dems)
        print(f"--- Batch {i+1} ---")
        print(f'  Node demand shape  (batch_size, p_size+1, 1)               : {dems.size()}')
        print(f'  Cost matrix shape  (batch_size, p_size+1, p_size+1)        : {cost_mat.size()}')
        print(f'  Node feature shape (batch_size, p_size+1, NODE_FEATURE_DIM): {feats.size()}')
# DEMO
kwargs = {
    "generator": generator,
    "num_sample": 80,
    "batch_size": 32,
    "rng": np.random.default_rng(42), # fix seed/rng for reproducible result
}
examples = [ # pairs of (dataloader, desciption)
    (DATALOADER(**kwargs), ' Default problem size (50) '),
    (DATALOADER(problem_sizes_mean=40, **kwargs), ' Fixed problem size '),
    (DATALOADER(problem_sizes_mean=40, problem_sizes_std=4, **kwargs), ' Random problem size from N(40,4^2) '),
    (DATALOADER(problem_sizes_mean=[35,40,45], **kwargs), ' Fixed problem size per batch '),
    (DATALOADER(problem_sizes_mean=[35,40,45], problem_sizes_std=[2,3,4], **kwargs), ' Random problem sizes per batch following the given means and stds '),
] # feel free to play around with the parameters

In [ ]:
# 🎯 TODO: Understand how to work with dataloaders and tensors
print("🗃️ Data Loader Usage Demo (Set example_id to 0, 1, 2, 3, or 4)\n")
example_id = 4 # 0,1,2,3,4 # Choose which example to display
data_loader_demo(*examples[example_id])

## 3.5 REINFORCE, Baselines & POMO

Afrer explaining all the setup, we are now ready to explain the main algorithm in this project.

POMO is a policy gradient method, meaning that the model itself is an actor which decides the next action based on the current state. This not so different from what you did to manually solve CVRP instances, but instead of a human brain, the model relies on "parametric knowledge" obtained from the training data.

More formally, we consider an actor network $\pi_\theta$ parametrized by $\theta$. When you vary the parameters $\theta$, the actor $\pi_\theta$ may take different actions from the same state.
By "training the actor" we mean optimize over $\theta$ so that the actor can maximize the expected return $J(\theta)$, i.e. minimize the expected solution cost.

The standard way to optimize over parameters is gradient descent, which you have already seen in Project 2. In order to carry this out, we need to know the gradient $\nabla_\theta J(\theta)$ of the objective function $J(\theta)$, but since we do not have an explicit expression for $J(\theta)$, we must estimate the gradient from our data. The REINFORCE algorithm uses the following estimator:

$$
\nabla_\theta J(\theta) \approx \frac{1}{M} \sum_{i=1}^M R(\tau_i) \nabla_\theta \log p_\theta(\tau_i),
$$
where $\tau_1,\ldots, \tau_M$ are some solution paths and $R(\tau_i)=-\text{cost}(\tau_i)$ is the return. The term $p_\theta(\tau_i)$ refers to the probability that actor $\pi_\theta$ samples the tour $\tau_i$, which can be computed by the conditional probability:
$$
p_\theta(\tau_i) := \prod_{k=0}^{|\tau_i|-1} \pi_\theta\left(\tau_i\text{[$k+1$]} \mid \tau_i\text{[:$k$]}\right),
$$
where $\tau_i[:k]$ is the list slicing operation in Python.

REINFORCE is the first foundational policy gradient algorithm, but this estimator is very unstable and can mislead your training process towards the wrong direction. This is because the reward signals are too sparse (only one per episode) and non-targeted: the actor may be doing well on some of the states but poorly on others, and there is no way to tell based on one single number! This is known as the Credit Assignment Problem, the fundamental challenge in RL of determining which specific actions, among a long sequence of actions, are responsible for a delayed reward. Researchers have proposed what are called "baselines" to address this problem. Intuitively speaking, baselines are quantities which you subtract from a return in order to estimate the "advantage" of certain actions, i.e. how good one action is when measured by some standards. Using this idea, the REINFORCE with baseline algorithm uses the following estimator instead:
$$
\nabla_\theta J(\theta) \approx \frac{1}{M} \sum_{i=1}^M \left(R(\tau_i)-b_i\right) \nabla_\theta \log p_\theta(\tau_i),
$$
where $b_1,\ldots. b_M$ are some numbers (possibly depending on the current state) called baselines. This algorithm has much lower variance, so the training process is a lot more stable and easier to converge to a good model.

The POMO algorithm is nothing but a REINFORCE algorithm with a particular baseline. A good baseline should inform the model how good the downstream actions are from a given state.
POMO uses the following baseline: Given an instance of size $N$, it considers $N$ paths $\tau_1,\ldots, \tau_N$ in parallel where $\tau_j$ is forced to start from customer node $j$. The baseline is then calculated as the average of the returns:
$$
b:=\frac{1}{N}\sum_{j=1}^N R(\tau_j).
$$


For book-keeping, we treat each $\tau_j$ as a row and stack them in a matrix to represent a "group state", and we refer to this environemnt as a "group environment". As an example, if an instance has $4$ customer nodes, a possible trajectory of states in an episode may look like this:

\begin{align*}
\begin{bmatrix}
\quad \\  \\  \\  \\   
\end{bmatrix}
& \xrightarrow[\text{forced}]{\text{Action 1}}
\begin{bmatrix}
0 \\ 0 \\ 0 \\ 0
\end{bmatrix}
\xrightarrow[\text{forced}]{\text{Action 2}}
\begin{bmatrix}
0 & 1 \\ 0 & 2 \\ 0 & 3 \\ 0 & 4
\end{bmatrix}
\xrightarrow[\text{e.g.}]{\text{Action 3}}
\begin{bmatrix}
0 & 1 & 4 \\ 0 & 2 & 1 \\ 0 & 3 & 0\\ 0 & 4 & 1
\end{bmatrix}
\xrightarrow[\text{e.g.}]{\text{Action 4}}
\begin{bmatrix}
0 & 1 & 4 & 3\\ 0 & 2 & 1 & 4\\ 0 & 3 & 0 & 4\\ 0 & 4 & 1 & 2
\end{bmatrix}
\xrightarrow[\text{e.g.}]{\text{Action 5}}
\begin{bmatrix}
0 & 1 & 4 & 3 & 2\\ 0 & 2 & 1 & 4 & 0\\ 0 & 3 & 0 & 4 & 0 \\ 0 & 4 & 1 & 2 & 3
\end{bmatrix} \\
& \xrightarrow[\text{e.g.}]{\text{Action 6}}
\begin{bmatrix}
0 & 1 & 4 & 3 & 2 & 0\\ 0 & 2 & 1 & 4 & 0 & 3\\ 0 & 3 & 0 & 4 & 0 & 1\\ 0 & 4 & 1 & 2 & 3 & 0
\end{bmatrix}
\xrightarrow[\text{e.g.}]{\text{Action 7}}
\begin{bmatrix}
0 & 1 & 4 & 3 & 2 & 0 & 0\\ 0 & 2 & 1 & 4 & 0 & 3 & 0\\ 0 & 3 & 0 & 4 & 0 & 1& 2\\ 0 & 4 & 1 & 2 & 3 & 0 & 0
\end{bmatrix}
\xrightarrow[\text{forced}]{\text{Action 8}}
\begin{bmatrix}
0 & 1 & 4 & 3 & 2 & 0 & 0& 0\\ 0 & 2 & 1 & 4 & 0 & 3 & 0& 0\\ 0 & 3 & 0 & 4 & 0 & 1& 2& 0\\ 0 & 4 & 1 & 2 & 3 & 0 & 0& 0
\end{bmatrix}
\end{align*}

Notice that the first and the second actions are forced: All tours must start from the depot, and the POMO algorithm requires the following action to cover all customer nodes. The remaining actions are decided by the actor, but when the tour for a row is complete, the only legal action to take is to stay at the depot. For example, Row 1 and Row 4 are finished after Action 6, Row 2 after Action 7, and Row 3 after Action 8, so rows which finish early are padded with zero for convenience of tensor operations.

Recall that our dataloader emits batches of random instances, and each batch can have possibly different instance sizes. This means one group state will be created for one batch, for which one baseline will be computed.

You can see a mock training process in action in the cell below, or open `visualizations/pomo_training.html` in your browser:

In [ ]:
# @title Visualize POMO Training Progress { display-mode: "form" }
import base64, pathlib
html = pathlib.Path("visualizations/pomo_training.html").read_text(encoding="utf-8")
b64 = base64.b64encode(html.encode()).decode()
HTML(f'<iframe src="data:text/html;base64,{b64}" '
     f'width="100%" height="920" style="border:1px solid #ddd;border-radius:8px"></iframe>')

The pseudocode of the POMO training procedure is given in the cell below. You may find it useful for implementation.

> <font color="red" size="6"><i><b>POMO TRAINING</i><b></font>
>
> ***Initialize*** Parameters $\theta$, learning rate schedule $\alpha_t>0$ $(t=1,2,3,\ldots)$
>
> For iteration $t=1,2,3,\ldots$ do the following:
> 1. Create a new dataloader which generate random instances
> 2. For each batch in dataloader do the following:
>    1) Let $N$ be the problem size for this batch and $B$ be the batch size
>    2) For each instance $j \in \{1,\ldots, B\}$ complete an episode in the group environment to get solutions $\tau^1_j,\ldots, \tau^N_j$ and compute the baseline:
>       $$b_j:=\frac{1}{N}\sum_{i=1}^N R(\tau^i_j)$$
>    3) Compute the gradient estimate:
>       $$\nabla_\theta J(\theta) \leftarrow \frac{1}{NB} \sum_{i=1}^N\sum_{j=1}^B \left(R(\tau^i_j)-b_j\right) \nabla_\theta \log p_\theta(\tau_i)$$
>    4) Perform gradient ascent: $\theta \leftarrow \theta + \alpha_t \nabla_\theta J(\theta)$

With these information, we are finally ready to start coding. In the rest of this section, you will need to implement the core parts of the POMO RL setup disucssed above.

# 🎯 Part 4: Implementations - Your Task

$\fbox{Big picture: how the code is organized}$

You are about to implement the RL side of POMO, including the state `GROUP_STATE`, the environment `GROUP_ENVIRONMENT` and the training logic.
<center>

```
               YOU IMPLEMENT                                ALREADY PROVIDED        
  ┌─────────────────────────────────────────┐      ┌───────────────────────────────┐
  │                                         │      │                               │
  │  GROUP_STATE ("where everyone is")      │      │  ACTOR  (grouped_actor.py)    │
  │  Tracks the following variables:        │      │                               |
  │  · node features                        │◀─────│  · encoder + decoder          │
  │  · progress of all tours                │reads │  · reads the current state    │
  │  · constraints (masks)                  │      │  · outputs P(next node)       │
  │                                         │      │                               │
  ├─────────────────────────────────────────┤      └───────────────┬───────────────┘
  │                                         │                      │                
  │  GROUP_ENVIRONMENT ("rules of the game")│                      │ action:        
  │  It has the following methods:          │◀─────────────────────┘ chosen next    
  │  · reset(): gives fresh initial state   │                        node per tour  
  │  · step(action): updates state, check   │                                       
  │     check if every tour is finished     │                                       
  │ · _get_travel_cost(): compute -return   │                                       
  │                                         │                                       
  └─────────────────────────────────────────┘                                       
```
</center>

One important thing to keep in mind: the GROUP_STATE class tracks `batch_size × group_size` tours at the same time using tensors with those two
leading dimensions. This allows for much faster parallel processing inside of `for` loop over tours.

## 🎯 4.1 Defining the (Group) State - Your Task

In [ ]:
from source.parameters import VEHICLE_CAPACITY

In [ ]:
# @title Implement the GROUP_STATE class { display-mode: "form" }
class GROUP_STATE:
    '''
    Here, we define the GROUP_STATE class to manage the state of multiple CVRP tours in parallel.
    You should think of a GROUP_STATE instance as a container that bundles group_size tours together
    for parallel processing, including intialization, updates, and masking of invalid actions
    violating CVRP constraints.

    The intput data tensor has shape (batch, problem_size+1, NODE_FEATURE_DIM) and contains demands and
    cost information for the depot and customers.

    We define a group state because in POMO, we launch multiple tours (group_size) for each problem
    instance in the batch simultaneously by taking different starting nodes, and these tours will
    give us the baseline for calculating the advantage during training. In training, we take
    group_size=problem_size.

    For each batch_index and group_index, there is a separate tour being tracked. This means each
    GROUP_STATE instance manages batch_size*group_size tours in total.
    '''

    def __init__(self, group_size, data, int_demand, vehicle_capacity=VEHICLE_CAPACITY):
        '''
        Initializes an instance of the GROUP_STATE class.
        Parameters:
            group_size: number of parallel tours per problem instance in the batch
            data: Tensor of shape (batch, problem+1, NODE_FEATURE_DIM) containing the CVRP instance data
            int_demand: LongTensor of shape (batch, problem+1, 1) containing integer-valued demand for each node
            vehicle_capacity: the capacity of each vehicle
        '''
        # We divide all the info to manage into Basic Info, History, and Status sections
        # ====================== Basic Info ======================
        self.batch_s = data.size(0) # batch size of the input data, i.e. how many random instances
        self.group_s = group_size   # group size, i.e. number of parallel tours per instance
        self.problem_size = data.size(1) - 1 # number of customers (excluding depot) for this batch
        self.data = data # node features, shape = (batch, problem+1, NODE_FEATURE_DIM)
        self.int_demand = int_demand # shape = (batch, problem+1, 1), integer-valued demand for each node
        self.vehicle_capacity = int(vehicle_capacity) # shape = (batch, problem+1, 1)

        # ======================= History =======================
        self.selected_count = 0     # how many actions have been taken so far
        self.current_node = None    # current node index for each tour; shape = (batch, group)

        # History of selected nodes for each tour; shape = (batch, group, selected_count)
        self.selected_node_list = torch.zeros((self.batch_s, self.group_s, 0), dtype=torch.long, device=DEVICE)

        # ======================= Status =======================
        # Whether each tour is at the depot; shape = (batch, group)
        self.at_depot = None

        # Remaining capacity; shape = (batch, group)
        # Initially, all vehicles are fully loaded (i.e. with value vehicle_capacity); will decrease when visiting customers
        
        # 🎯 TODO: Create a torch.LongTensor with the correct value and shape
        # Hint: Use torch.ones(shape_tuple, dtype=torch.long, device=...)
        self.remaining_capacity = ...

        # Whether each node has been visited in each tour; shape = (batch, group, problem+1)
        # Initially, all nodes are unvisited (0); will be set to -inf when a node is visited
        # 🎯 TODO: Create a torch.FloatTensor of zeros with the correct shape
        # Hint: Use torch.zeros(shape_tuple, dtype=torch.float32, device=...)
        self.visited_ninf_flag = ...
        
        # A mask with -inf for invalid next nodes and 0 for valid nodes; shape = (batch, group, problem+1)
        # 🎯 TODO: Create a torch.FloatTensor of zeros with the correct shape
        # Hint: Use torch.zeros(shape_tuple, dtype=torch.float32, device=...)
        self.ninf_mask = ...

        # Whether each tour has finished (initialized to False); shape = (batch, group)
        # 🎯 TODO: Create a torch.BoolTensor with the correct values and shape
        # Hint: Use dtype=torch.bool for a BoolTensor
        self.finished = ...

    def move_to(self, selected_idx_mat):
        '''
        Update the group state based on the selected next nodes for each tour.
        Parameters:
            selected_idx_mat: Tensor of shape (batch, group) containing the indices of the next
                              nodes selected for each tour in the batch.
        '''

        # ======================= History =======================
        # 1. Increment the number of actions taken by one
        self.selected_count = ... # 🎯 TODO: increment selected_count by 1

        # 2. Update current node for each tour being tracked by this GROUP_STATE instance
        self.current_node = selected_idx_mat
        # 3. Update the history of selected nodes by appending the newly selected nodes
        self.selected_node_list = torch.cat((self.selected_node_list, selected_idx_mat[:, :, None]), dim=2)

        # ======================= Status =======================
        # 1. Mark whether each tour is currently at the depot
        # 🎯 TODO: update at_depot based on whether selected_idx_mat is 0
        # Hint: (selected_idx_mat == 0) is a BoolTensor
        self.at_depot = ...

        # 2. Fetch the demand of the selected nodes
        demand_list = self.int_demand[:, None, :, 0].expand(self.batch_s, self.group_s, -1) # shape = (batch, group, problem+1)
        gathering_index = selected_idx_mat[:, :, None] # shape = (batch, group, 1)
        selected_demand = demand_list.gather(dim=2, index=gathering_index).squeeze(dim=2) # shape = (batch, group)

        # 3. Update the remaining capacity based on the selected_demands
        # 🎯 TODO: subtract selected_demand from remaining capacity of each tour
        self.remaining_capacity = ...

        # 4. Refill capacity if at the depot
        # Student Version:
        # 🎯 TODO: set remaining capacity to vehicle_capacity for tours that are at the depot
        # Hint: Use at_depot as a boolean mask for indexing into remaining_capacity
        self.remaining_capacity ...

        # 5. Set visited flag for selected nodes to -inf
        batch_idx_mat = torch.arange(self.batch_s)[:, None].expand(self.batch_s, self.group_s)
        group_idx_mat = torch.arange(self.group_s)[None, :].expand(self.batch_s, self.group_s)
        self.visited_ninf_flag[batch_idx_mat, group_idx_mat, selected_idx_mat] = -float('inf')

        # 6. Update finished status: a tour is finished if all nodes have been visited
        self.finished = self.finished  | torch.isneginf(self.visited_ninf_flag).all(dim=2)
        # shape = (batch, group)

        # ======================= Status Edit =======================
        # 1. Update self.visited_ninf_flag so that Status 6 in the next iteration is correct
        # Allow visit to depot anytime except those currently at the depot
        self.visited_ninf_flag[:, :, 0] = torch.where(self.at_depot, self.visited_ninf_flag[:, :, 0], 0.0)

        # 2. Update self.ninf_mask
        # Check which nodes are illegal to visit for the next step based on CVRP constraints
        # # 2a. Identify nodes with demand too large for the remaining capacity
        demand_too_large = self.remaining_capacity[:, :, None] < demand_list
        # This is Torch.BoolTensor with shape = (batch, group, problem+1)

        # 2b. First, start with the visited nodes since they cannot be visited again
        self.ninf_mask = self.visited_ninf_flag.clone()

        # 2c. Next, mask nodes whose demand exceeds remaining capacity
        self.ninf_mask ... # TODO: set -inf for nodes with demand too large

        # 2d. Unmask the depot (index 0) for those finished tours (they were masked by 1 and 2a)
        self.ninf_mask[:, :, 0] = torch.where(self.finished, 0.0, self.ninf_mask[:, :, 0])

# Monkey-patching: Replace existing cvrp.GROUP_STATE with your implementation
cvrp.GROUP_STATE = GROUP_STATE
print("✅ GROUP_STATE class implemented. Your implementation will be used moving forward.")

In [ ]:
# @title ✅ Self-check for your GROUP_STATE implementation { display-mode: "form" }
# These tests run on tiny hand-made inputs and print a hint for anything wrong.
from source.student_tests import test_group_state
test_group_state(GROUP_STATE, DEVICE)

## 🎯 4.2 Defining the (Group) Environment - Your Task

In [ ]:
# @title Implement the GROUP_ENVIRONMENT class { display-mode: "form" }
class GROUP_ENVIRONMENT:
    '''
    This class manages the group environment for CVRP, handling multiple tours in parallel.
    Think of this as the RL environment we discussed above with GROUP_STATE as states.
    It initializes the environment, resets it for new groups of tours, steps through the
    environment based on selected actions, and calculates rewards when all tours are finished.
    '''

    def __init__(self, all_demands, all_features, cost_matrix, vehicle_capacity=VEHICLE_CAPACITY):
        '''
        Initializes the GROUP_ENVIRONMENT with CVRP instance data.
        Parameters:
            all_demands: LongTensor of shape (batch, problem+1, 1)
            all_features: FloatTensor of shape (batch, problem+1, NODE_FEATURE_DIM)
            cost_matrix: FloatTensor of shape (batch, problem+1, problem+1) with pairwise travel costs
            vehicle_capacity: int, the capacity of each vehicle
        '''

        self.batch_s = all_demands.size(0) # batch size
        self.group_s = None # group size will be set during in the reset method
        self.group_state = None # will hold the current group state

        self.int_demand = all_demands
        self.data = all_features # shape = (batch, problem+1, NODE_FEATURE_DIM)
        self.cost_matrix = cost_matrix
        self.vehicle_capacity = int(vehicle_capacity)

    def reset(self, group_size):
        '''
        Resets the environment for a new group of tours. Think of this as starting
        a new episode in this RL environment. This entails creating a new GROUP_STATE
        instance as the initial state, as well as setting up variables to track the
        rewards and done status.

        Parameters:
            group_size: number of parallel tours per problem instance in the batch
        '''
        self.group_s = group_size # set group size
        self.group_state = GROUP_STATE(group_size=group_size, data=self.data,
                                       int_demand=self.int_demand,
                                       vehicle_capacity=self.vehicle_capacity) # initialize new group state

        reward = None # no reward at the beginning
        done = False # whether all tours are finished, i.e. is the episode over?
        return self.group_state, reward, done

    def step(self, selected_idx_mat):
        '''
        Make a step in the environment based on the selected next nodes for each tour.
        Parameters:
            selected_idx_mat: Tensor of shape (batch, group) containing the indices of
                              the next nodes selected for each tour in the batch.
        Returns:
            group_state: updated GROUP_STATE instance after the step
            reward: reward obtained after the step; None if not done; otherwise a tensor
                    of shape (batch, group) containing negative tour distances
            done: boolean indicating whether all tours are finished
        '''

        # 1. Perform the action by updating the group state
        # 🎯 TODO: update self.group_state based on selected_idx_mat
        # Hint: use the appropriate method of this group_state object
        self.group_state ...

        # 2. Check if all tours are finished and update the done status
        # Student Version:
        # 🎯 TODO: set done to whether all tours are finished
        # Hint: self.group_state.finished is a torch.BoolTensor of shape (batch, group)
        #       indicating which tours are finished. Convert this to a single boolean
        #       using BoolTensor.all()
        done = ...

        # 3. Calculate reward if all tours are finished, shape = (batch, group)
        if done:
            # 🎯 TODO: calculate negative total travel cost for each tour
            # Hint: use the appropriate method of this object
            reward = ...
        else:
            reward = None

        return self.group_state, reward, done

    def _get_travel_cost(self):
        '''
        Calculate the total travel cost for each tour in the group state once all tours are finished
        using the precomputed cost matrix.
        Returns:
            travel_costs (torch.FloatTensor): total travel cost for each tour; shape = (batch, group)
        '''
        # 1. Get the sequence of visited nodes, shape = (batch, group, selected_count)
        # 🎯 TODO: get the sequence of visited nodes from self.group_state
        # Hint: retrieve the appropriate attribute of the group_state object
        from_nodes = ...

        selected_count = from_nodes.size(2)

        # 2. Shift from_nodes by one step to the left to get the destination nodes
        to_nodes = from_nodes.roll(dims=2, shifts=-1)   # shape = (batch, group, selected_count)

        # 3. Create a batch index tensor to map back to the correct problem instance
        batch_idx = torch.arange(self.batch_s, device=from_nodes.device)[:, None, None].expand(
            self.batch_s, self.group_s, selected_count
        )

        # 4. Look up the pairwise costs from from_nodes to to_nodes in the cost_matrix
        segment_costs = self.cost_matrix[batch_idx, from_nodes, to_nodes] # shape = (batch, group, selected_count)

        # 5. Sum the costs over the entire tour sequence
        # 🎯 TODO: sum segment_costs over the selected_count dimension to get 
        #          the total cost for each tour
        # Hint: use tensor.sum(dim=...)
        travel_costs = ...

        return travel_costs

# Monkey-patching: Replace existing cvrp.GROUP_ENVIRONMENT with your implementation
cvrp.GROUP_ENVIRONMENT = GROUP_ENVIRONMENT
print("✅ GROUP_ENVIRONMENT class implemented. Your implementation will be used moving forward.")

In [ ]:
# @title ✅ Self-check for your GROUP_ENVIRONMENT implementation { display-mode: "form" }
# These tests run on tiny hand-made inputs and print a hint for anything wrong.
from source.student_tests import test_group_environment
test_group_environment(GROUP_ENVIRONMENT, DEVICE)

## 4.3 Importing the Actor Class

The actor class subsumes two neural networks: the Transformer Encoder and the Decoder (called `NextNodeProbabilityCalculator` in the code), as explained in Section 3.
Below we import the code for future use.

<span style="color: red; font-weight: bold; font-style: italic; font-size: 20px;">
WARNING: You must implement the classes above before running the import cell below.
</span>
Otherwise, you need to restart the notebook kernel.

In [ ]:
from source.grouped_actor import ACTOR

## 🎯 4.4 Defining the Training Logic - Your Task

In [ ]:
# @title Import Libraries for Training & Evaluation { display-mode: "form" }
import time # for time tracking
from logging import Logger # for logging training progress and evaluation results
import source.evaluate_grouped_actors as EvaluateModule
from source.utilities import Average_Meter # for tracking averages
EvaluateModule.DEVICE = DEVICE # Set device in evaluation module for correct tensor allocation

In [ ]:
# @title Implement the TRAIN function { display-mode: "form" }
def TRAIN(grouped_actor: ACTOR,
          generator: CVRPGenerator,
          epoch: int,
          timer_start: float,
          logger: Logger,
          TRAIN_DATASET_SIZE: int,
          BATCH_SIZE: int,
          LOG_PERIOD_SEC: int,
          problem_sizes_mean=None,
          problem_sizes_std=None,
          rng=None):
    ''' Train the grouped actor for one epoch using REINFORCE with baseline.
    Parameters:
        grouped_actor: ACTOR instance to be trained
        generator: CVRPGenerator instance for generating training data
        epoch: current epoch number (an epoch means one full pass through the training dataset)
        timer_start: start time of the epoch for logging
        logger: logger instance for logging training progress
        TRAIN_DATASET_SIZE: total number of training samples in the dataset
        BATCH_SIZE: batch size for training
        LOG_PERIOD_SEC: logging period in seconds
        problem_sizes_mean: means of problem sizes used for training, (list of) numbers
        problem_sizes_std: stds of problem sizes used for training, (list of) nonnegative numbers
        rng: random number generator
    Remark:
        If problem_sizes_mean is None, the dataloader uses default value 25 duplicated across batches.
        If problem_sizes_std is None, the dataloader uses default value 0 duplicated across batches.
        The parameters problem_sizes_mean and problem_sizes_std are thus (eventually) lists of length
        ceil(TRAIN_DATASET_SIZE/BATCH_SIZE) i.e. number of batches. Thus each batch is associated with
        sampling parameters mu and sigma, and the problem size for that batch is given by rounding a
        gaussian variable with mean mu and std sigma, clipped to the valid range of [5,40].
    '''

    # Set up for training
    grouped_actor.train() # set the actor to training mode
    device = grouped_actor.device # device to use for training
    cost_AM = Average_Meter(device) # for logging average cost
    actor_loss_AM = Average_Meter(device) # for logging average actor loss

    # Set up AMP (Automatic Mixed Precision) for faster training on GPU
    if device == 'cuda':
        # L4/A100/H100 support bfloat16 natively. T4 supports float16 better.
        gpu_name = torch.cuda.get_device_name(device)
        amp_dtype = torch.bfloat16 if ("L4" in gpu_name or "A100" in gpu_name or "H100" in gpu_name) else torch.float16
        use_scaler = True
    elif device == 'mps':
        amp_dtype = torch.float16  # Mac MPS supports float16 acceleration
        use_scaler = False         # MPS doesn't require/support CUDA GradScaler
    else:
        amp_dtype = torch.float32  # CPU training stays standard float32
        use_scaler = False
    # Initialize scaler conditionally
    scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
    # print(f"⚡ AMP setup for {device}: amp_dtype={amp_dtype}, use_scaler={use_scaler}")

    # Create data loader for training dataset
    train_loader = DATALOADER(generator=generator,
                              num_sample=TRAIN_DATASET_SIZE,
                              batch_size=BATCH_SIZE,
                              problem_sizes_mean=problem_sizes_mean,
                              problem_sizes_std=problem_sizes_std,
                              rng=rng)

    # Start training loop over batches
    logger_start = time.time()
    episode = 0 # total number of samples processed so far
    for demands, cost_matrix in train_loader: # loop over batches
        # One training iteration (gradient update) will be performed for this batch below
        # Shapes of these tensors:
        # demands.shape = (batch, problem+1, 1)
        # cost_matrix.shape = (batch, problem+1, problem+1)
        batch_s = demands.size(0) # batch size
        group_s = demands.size(1)-1  # group size = problem size
        episode = episode + batch_s # update number of samples processed

        # Move tensors to the correct device for training
        demands = demands.to(device)
        cost_matrix = cost_matrix.to(device)
        # Compute features, shape = (batch, problem+1, NODE_FEATURE_DIM)
        features = cvrp.compute_batched_features(generator, cost_matrix, demands) # already on device
        
        # Dynamically uses 'cuda', 'cpu', or 'mps' and the correct precision type
        with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device!='cpu')):
            # 1. Initialize Environment and Grouped Actor for this batch
            # 1a. Create environment
            # Student Version:
            # 🎯 TODO: create GROUP_ENVIRONMENT instance based on input data
            # Hint: the required input is given by the constructor, i.e. __init__
            env = ...
            
            # 1b. Reset environment based on input data to get initial group state, reward (None), done (False)
            group_state, reward, done = env.reset(group_size=group_s)
            grouped_actor.reset(group_state, env)

            # 2. Make the first move
            # 2a. First moves are given: all routes must start from depot (node 0)
            # 🎯 TODO: Create a LongTensor of shape (batch, group) filled with 0 on device
            first_action = ...
            
            # 2b. Make a step in the environment based on first_action
            # 🎯 TODO: make a step in env based on first_action
            # Hint: Call the appropriate method of the env object
            group_state, reward, done = ...

            # 3. Make the second move
            # 3a. Second moves are given: each route must visit a unique node (1 to problem size)
            #     in order to compute the baseline for REINFORCE
            second_action = torch.arange(1, group_s+1, dtype=torch.long, device=device)[None, :].expand(batch_s, group_s)
            # 3b. Make a step in the environment based on second_action
            # 🎯 TODO: make a step in env based on second_action
            group_state, reward, done = ...

            # 4. Continue the group moves until done
            # 4a. Initialize group_prob_list to store the action probabilities at each step
            group_prob_list = torch.zeros((batch_s, group_s, 0), dtype=features.dtype, device=device)
            # 4b. Define batch and group index matrices for gathering chosen action probabilities
            batch_idx_mat = torch.arange(batch_s, dtype=torch.long, device=device)[:, None].expand(batch_s, group_s)
            group_idx_mat = torch.arange(group_s, dtype=torch.long, device=device)[None, :].expand(batch_s, group_s)
            
            while not done: # keep moving until all routes are finished
                # 4c. Get action probabilities from the grouped actor based on current group_state
                action_probs = grouped_actor.get_action_probabilities(group_state)
                # shape = (batch, group, problem+1)

                # 4d. Sample an action per tour based on the action probabilities
                action = action_probs.reshape(batch_s*group_s, -1).multinomial(1)\
                    .squeeze(dim=1).reshape(batch_s, group_s)
                # shape = (batch, group)

                # ==========================================
                # 4d': GUARDRAIL. On mac, torch.multinomial might pick an index that is masked out with -inf
                # due to numerical issues, which causes the environment step to fail. We override the sampled
                # action with the greedy action (argmax) for those tours to allow training to continue.
                is_invalid_action = torch.isneginf(group_state.ninf_mask[batch_idx_mat, group_idx_mat, action])
                if is_invalid_action.any():
                    greedy_action = action_probs.argmax(dim=2)
                    action = torch.where(is_invalid_action, greedy_action, action)
                # ==========================================

                # 4e. Make a step based on the sampled actions
                action[group_state.finished] = 0  # force finished routes to stay at depot
                # 🎯 TODO: make a step in env based on action
                group_state, reward, done = ...

                # 4f. Record the chosen action probabilities to perform gradient updates later
                # Gather the probabilities of the chosen actions; shape = (batch, group)
                chosen_action_prob = action_probs[batch_idx_mat, group_idx_mat, action].reshape(batch_s, group_s)
                chosen_action_prob[group_state.finished] = 1  # done episode will gain no more probability
                # Concat existing and new probabilities along the action sequence dimension; shape = (batch, group, selected_count)
                group_prob_list = torch.cat((group_prob_list, chosen_action_prob[:, :, None]), dim=2)


            # 5. Learning: actor update using REINFORCE with baseline
            # 5a. Get the group rewards from the environment
            group_reward = reward # shape = (batch, group)

            # 5b. Compute the log probabilities of the taken actions
            group_log_prob = group_prob_list.log().sum(dim=2) # shape = (batch, group)

            # 5c. Compute the advantages using baseline, where baseline = mean reward across the group for each sample
            # 🎯 TODO: compute advantage by subtracting mean group reward from group_reward
            # Hint: use the method tensor.mean(dim=..., keepdim=...) to compute mean across the group dimension
            baseline = ...
            group_advantage = ...

            # 5d. Compute the policy gradient loss
            # This is Step 2C in the training algorithm from Section 4.2
            # First, compute the loss for every tour and store it in the tensor
            # group_loss of shape (batch, group)
            # 🎯 TODO: compute the policy gradient loss for each tour
            # Hint: the loss is -advantage * log_prob
            group_loss = ...

            # Next, compute the mean loss across all tours in the batch
            # 🎯 TODO: compute the average loss for this group state
            # Hint: Use tensor.mean()
            loss = ...

        # 5e. Backpropagation and optimizer step
        # This is Step 2D in the training algorithm from Section 4.2
        grouped_actor.optimizer.zero_grad()
        # If scaler is disabled, it acts as a transparent pass-through
        scaler.scale(loss).backward()
        scaler.step(grouped_actor.optimizer)
        scaler.update()

        # 6. Recording for logging
        max_reward, _ = group_reward.max(dim=1) # max reward = - min distance
        cost_AM.push(-max_reward)  # reward was given as negative dist
        actor_loss_AM.push(group_loss.detach()) # detach to disable gradient tracking

        # 7. Logging
        if (time.time()-logger_start > LOG_PERIOD_SEC) or (episode == TRAIN_DATASET_SIZE):
            timestr = time.strftime("%H:%M:%S", time.gmtime(time.time()-timer_start))
            log_str = 'Ep:{:03d}-{:07d}({:5.1f}%)  T:{:s}  Loss:{:+5f}  AvgCost:{:5f}' \
                .format(epoch, episode, episode/TRAIN_DATASET_SIZE*100,
                        timestr, actor_loss_AM.result(), cost_AM.result())
            logger.info(log_str)
            logger_start = time.time()

    # 8. Update the learning rate scheduler after each epoch
    grouped_actor.lr_stepper.step()

# 🎯 Part 5: Training - Your Task


Training a deep learning model is not as easy as it may sound even if you have implemented the training logic above.

During the training process, you need to monitor the performance of your model. This is typically done on a held-out test set, but since we are training with randomly generated data, we simply generate another dataset for evaluation. Since we are concerned with our day-to-day delivery operations where the number of customers are random, we compute the evaluation score as the average solution distance across all possible instance sizes. In other words, a single model needs to perform well regardless of the input size. Naively, you might just randomly sample input sizes and start training, but since small and large instance can be very different by nature, the model can "get confused" and fail to learn if you don't generate your data in a proper manner. For example, you may first train the model to solve small instances before moving on to big ones. This is called *curriculum learning*, in which the task difficulty is increased gradually during training.

Another big part of training models concerns *hyperparameter tuning*. While parameters refer to internal variables that the model learns from data during training, hyperparameters refer to external configuration variables that the user sets before training begins to guide the learning process. Examples include the number of layers, the dimension of embeddings, and the multiplier for regularization. Hyperparameter tuning is vital to achieve a good performance and can be computationally expensive.

Below we will go through the process step by step.

## 5.1 Hyperparameters for RL Model Architecture & Training

Here we set up the configurations for the model architecture and training.
We recommend that you use the default values provided to check if your implementations above are correct.

Note: The `LOG_PERIOD_EPOCH` defines how often you save a checkpoint (snapshot) of your model during training. You can load saved checkpoints for evaluation later on, but do not save checkpoints too often as they take up space.


In [ ]:
# @title Hyperparameters { display-mode: "form" }
from dataclasses import dataclass, asdict # for dataset manipulation
from source.parameters import MIN_NUM_CUSTOMERS, MAX_NUM_CUSTOMERS
from typing import Tuple
P_SIZES = list(range(MIN_NUM_CUSTOMERS, MAX_NUM_CUSTOMERS + 1))
# 🎯 TODO: Tune the following hyperparameters for better performance
D_EPOCHS = list(range(5, 26, 5)) + list(range(30, 60, 2))
@dataclass(frozen=True)
class HyperParams:
    # Epochs and Dataset sizes
    # Batch size: smaller means nosier gradient estimates, larger means more computation per step
    # Note: More data generally leads to better results but training takes longer
    # Note: The loss should stabilize after an adequate number of epochs.
    #       If not, consider increasing TOTAL_EPOCH.
    TOTAL_EPOCH: int = 10 # 60                  # e.g. 50, 100, ...
    BATCH_SIZE: int = 128                       # training batch size; e.g. 32, 64, ...
    TRAIN_DATASET_SIZE: int = 128*50 # 128*400  # total training data size; increase this as needed
    TEST_BATCH_SIZE: int = 128                  # eval/test batch size, no need to change
    TEST_DATASET_SIZE: int = 128*len(P_SIZES)   # total eval/test data size, no need to change

    # Architecture Hyper-Parameters
    # Note: Larger values give more expressive model but they harder to train
    # Bigger models are not always better; tune based on validation performance
    EMBEDDING_DIM: int = 128                    # e.g. 64, 128, 256, 512
    KEY_DIM: int = 16                           # Length of q, k, v of EACH attention head
    HEAD_NUM: int = 8                           # Number of attention heads
    ENCODER_LAYER_NUM: int = 6                  # Number of encoder layers
    FF_HIDDEN_DIM: int = 256                    # e.g. 128, 256, 512
    LOGIT_CLIPPING: float = 10                  # C in the Kool et al paper

    # Learning Rate Hyper-Parameters
    ACTOR_LEARNING_RATE: float = 5e-4           # initial learning rate for the actor; e.g. 1e-4, 5e-4, 1e-3
    ACTOR_WEIGHT_DECAY: float = 1e-6            # L2 regularization coefficient, e.g. 1e-6, 1e-5, 1e-4
    LR_DECAY_EPOCH: Tuple = tuple(D_EPOCHS)     # which epochs to decay learning rate
    LR_DECAY_GAMMA: float = 0.8                 # multiply learning rate by this factor on decay

    # Logging
    LOG_PERIOD_SEC: int = 30                    # log progress every n seconds
    LOG_PERIOD_EPOCH: int = 10                  # save model checkpoint every n epochs

# Initialize hyperparameters
PARAMS = HyperParams()
# Separate hyperparameters by their usage for easier passing to functions
extract_hyperparms = lambda keys: {k: v for k, v in asdict(PARAMS).items() if k in keys}
CONFIG_MODEL = extract_hyperparms(['EMBEDDING_DIM', 'ENCODER_LAYER_NUM', 'HEAD_NUM', 'KEY_DIM', 'FF_HIDDEN_DIM', 'LOGIT_CLIPPING'])
CONFIG_TRAIN = extract_hyperparms(['TRAIN_DATASET_SIZE', 'BATCH_SIZE', 'LOG_PERIOD_SEC'])
CONFIG_EVAL = extract_hyperparms(['TEST_DATASET_SIZE', 'TEST_BATCH_SIZE'])

print("🎛️ Hyperparameters initialized successfully. Experiment with different configurations for better results.\n")
print('Model hyperparameters:', CONFIG_MODEL)
print('Training hyperparameters:', CONFIG_TRAIN)
print('Evaluation hyperparameters:', CONFIG_EVAL)
print("Learning rate decay schedule:", PARAMS.LR_DECAY_EPOCH, "with gamma =", PARAMS.LR_DECAY_GAMMA)

## 5.2 Logging Setup

In [ ]:
# @title Set up logging for training { display-mode: "form" }
# WARNING: The Get_Logger method creates a new folder based on the current timestamp per execution
# The folder name will be prefixed with timestamp and saved under 'results' folder
# e.g. results/20260619_2130__TRAIN
# Execute this cell only once per training run unless you intend to create new folders
import json # for JSON file operations
from source.utilities import Get_Logger   # for logging

SAVE_FOLDER_NAME_TRAIN = "TRAIN" # folder to save training logs and model checkpoints
logger, result_folder_path = Get_Logger(SAVE_FOLDER_NAME_TRAIN)

# Save the hyperparameters used for this run to colab virtual machine
hyper_param_save_path = '{}/used_HYPER_PARAMS.json'.format(result_folder_path)
with open(hyper_param_save_path, 'w') as f:
    json.dump(asdict(PARAMS), f, indent=4)
# Copy the folder to drive to persist
vm_folder = PROJECT_DIR + result_folder_path[1:] # remove '.' prefix from result_folder_path
drive_folder = DRIVE_DIR + result_folder_path[1:]
shutil.copytree(vm_folder, drive_folder, dirs_exist_ok=True)

print("💾 Logging has been set up in folder:", vm_folder, "and copied to:", drive_folder)

## 5.3 Initialize the RL Model (Actor) & Components for Training


Besides the actor model itself, we also need an optimizer and a learning rate scheduler for training.
The optimizer does a fancy version of gradient ascent and tries to maximize the return by updating the model weights. The updates are done incrementally and the extent to which updates happen is controlled by the learning rate scheduler.

In [ ]:
import torch.optim as optim # for optimization
import torch.optim.lr_scheduler as lr_scheduler  # for learning rate scheduling

In [ ]:
# @title Initialize the model and optimizer { display-mode: "form" }
actor = ACTOR(device=DEVICE, **CONFIG_MODEL)
assert (DEVICE == next(actor.encoder.parameters()).device.type) and (DEVICE == next(actor.node_prob_calculator.parameters()).device.type), \
        "Actor model parameters are not on the correct device!"

# Initialize optimizer for the actor
actor.optimizer = optim.Adam(actor.parameters(), lr=PARAMS.ACTOR_LEARNING_RATE, weight_decay=PARAMS.ACTOR_WEIGHT_DECAY)
# Initialize learning rate scheduler for the actor
actor.lr_stepper = lr_scheduler.MultiStepLR(actor.optimizer, milestones=PARAMS.LR_DECAY_EPOCH, gamma=PARAMS.LR_DECAY_GAMMA)

print("🤖 Actor model initialized and moved to device:", actor.device)
print("🎯 Optimizer initialized successfully")
print("📉 Learning rate scheduler initialized successfully")

If you have saved model/optimizer/lr_stepper state dicts from a previous run, you can continue traning from that checkpoint by using the `load_from_checkpoint` function below. Create a new folder to avoid overwritting the old version.

In [ ]:
# @title Helper function to load model, optimizer, and lr_stepper from a checkpoint { display-mode: "form" }
def load_from_checkpoint(training_round, checkpoint_epoch, root_dir=PROJECT_DIR):
    '''
    Utility function to load model, optimizer, and lr_stepper state dicts from a specific checkpoint.
    Parameters:
        training_round: name of the training round
        checkpoint_epoch: epoch number of the checkpoint to load
        root_dir: where the checkpoint is saved; should be PROJECT_DIR or DRIVE_DIR
    Returns:
        loaded_actor: ACTOR instance with loaded state dict
    '''
    assert root_dir in [PROJECT_DIR, DRIVE_DIR], "root_dir must be either PROJECT_DIR or DRIVE_DIR"
    # Load saved files
    actor_model_folder = f'{root_dir}/result/{training_round}/{checkpoint_epoch}'
    actor_model_path = f'{actor_model_folder}/ACTOR_state_dic.pt'
    optimizer_save_path = f'{actor_model_folder}/OPTIM_state_dic.pt'
    lr_stepper_save_path = f'{actor_model_folder}/LRSTEP_state_dic.pt'
    actor_config_path = f'{root_dir}/result/{training_round}/used_HYPER_PARAMS.json'
    with open(actor_config_path, 'r') as f:
        used_hyperparams = json.load(f)

    # Reconstruct the actor
    actor_hyperparams = {k: v for k, v in used_hyperparams.items() if k in ['EMBEDDING_DIM', 'ENCODER_LAYER_NUM', 'HEAD_NUM', 'KEY_DIM', 'FF_HIDDEN_DIM', 'LOGIT_CLIPPING']}
    actor = ACTOR(device=DEVICE, **actor_hyperparams).to(DEVICE) # WARNNIG: You must use the same CONFIG_MODEL as in training
    actor.load_state_dict(torch.load(actor_model_path, map_location=DEVICE))

    # Initialize optimizer and learning rate scheduler for the actor
    actor.optimizer = optim.Adam(actor.parameters(), lr=used_hyperparams['ACTOR_LEARNING_RATE'], weight_decay=used_hyperparams['ACTOR_WEIGHT_DECAY'])
    actor.lr_stepper = lr_scheduler.MultiStepLR(actor.optimizer, milestones=used_hyperparams['LR_DECAY_EPOCH'], gamma=used_hyperparams['LR_DECAY_GAMMA'])

    actor.load_state_dict(torch.load(actor_model_path, map_location=DEVICE))
    actor.optimizer.load_state_dict(torch.load(optimizer_save_path, map_location=DEVICE))
    actor.lr_stepper.load_state_dict(torch.load(lr_stepper_save_path, map_location=DEVICE))

    print(f"✅ Successfully loaded actor model, optimizer, and lr_stepper from checkpoint: {actor_model_folder}")
    return actor

In [ ]:
# # Uncomment this cell in case you want to resume training from a previous run
# # Load from PROJECT_DIR when possible as it will be faster
# which_training_round, which_checkpoint = "20260615_1845__TRAIN", "CheckPoint_ep00020"
# actor = load_from_checkpoint(which_training_round, which_checkpoint, root_dir=PROJECT_DIR)

## 5.4 Run the Main Training Loop & Record Evaluation Results

We finally have everything set up and are now ready to train the model. To do this, simply run the cell below. There is nothing you need to implement, but you need to keep an eye on the evaluation score after each epoch during training. Since we want our model to perform well on all possible number of customers, the evaluation is done with `TEST_BATCH_SIZE` random instance per customer size. In general, you should see the average cost in evaluation decrease over the course your training epochs.
There would be some fluctation due to our data being randomly generated, but if the data size is large enough this should not be an issue.

If the training behavior is not heading towards this direction, chances are that there's something wrong with your implementation, or you chose a poor combination of hyperparameters. In that case, go back to the previous cells and check if everything works as intended and/or change the hyperparameters.



In [ ]:
from matplotlib import ticker # for customizing plot ticks
from source.utilities import Extract_from_LogFile # for extracting results from log files

In [ ]:
# @title TRAINING LOOP { display-mode: "form" }
# Set up which epochs to save checkpoints (i.e. snapshots of the model)
checkpoint_epochs = np.arange(0, PARAMS.TOTAL_EPOCH+1, PARAMS.LOG_PERIOD_EPOCH)
# Set up rng for reproducibility
train_seed = 7
train_rng = np.random.default_rng(train_seed)

print(f"\n🚀 Training POMO for {PARAMS.TOTAL_EPOCH} epochs with batch size {PARAMS.BATCH_SIZE} and {PARAMS.TRAIN_DATASET_SIZE} training samples per epoch...\n")
timer_start = time.time() # get the current time
EvaluateModule.eval_result = [] # reset eval_result list before training
for epoch in range(1, PARAMS.TOTAL_EPOCH+1): # loop over epochs
    print(f"🧮 Starting epoch #{epoch} ... ")
    # For tracking purposes
    log_package = { 'epoch': epoch, 'timer_start': timer_start, 'logger': logger}

    # 🎯 TODO: Define the parameters for sampling problem sizes in this epoch
    # The values provided below is a good starting point. You may experiment with different
    # values (as a function of epoch number) to improve performance and/or speed up training.
    # Hint: For training stability, fix the problem size for the early epochs and gradually 
    #       increase the variation.
    problem_sizes_mean = 60
    problem_sizes_std = 0 # max(0, epoch-40)

    # Train the actor for one epoch (on random instance sizes)
    TRAIN(actor, generator, **log_package, **CONFIG_TRAIN,
          problem_sizes_mean=problem_sizes_mean,
          problem_sizes_std=problem_sizes_std,
          rng=train_rng)

    # Evaluate the actor after training for this epoch using random instances for each problem size in P_SIZES
    EvaluateModule.EVAL(actor, generator, **log_package, **CONFIG_EVAL, problem_sizes_mean=P_SIZES, rng=train_rng)

    # Save checkpoints
    if epoch in checkpoint_epochs:
        print(f"💾 Saving model checkpoint for epoch #{epoch} ...")
        checkpoint_folder_path = '{}/CheckPoint_ep{:05d}'.format(result_folder_path, epoch)
        os.mkdir(checkpoint_folder_path)

        model_save_path = '{}/ACTOR_state_dic.pt'.format(checkpoint_folder_path)
        torch.save(actor.state_dict(), model_save_path)
        optimizer_save_path = '{}/OPTIM_state_dic.pt'.format(checkpoint_folder_path)
        torch.save(actor.optimizer.state_dict(), optimizer_save_path)
        lr_stepper_save_path = '{}/LRSTEP_state_dic.pt'.format(checkpoint_folder_path)
        torch.save(actor.lr_stepper.state_dict(), lr_stepper_save_path)

        # Copy the checkpoint folder and the current log file to drive to persist
        vm_checkpoint_folder = PROJECT_DIR + checkpoint_folder_path[1:] # remove '.' prefix from checkpoint_folder_path
        drive_checkpoint_folder = DRIVE_DIR + checkpoint_folder_path[1:]
        shutil.copytree(vm_checkpoint_folder, drive_checkpoint_folder, dirs_exist_ok=True)
        log_file_path = result_folder_path[1:] + '/log.txt'
        shutil.copy2(PROJECT_DIR + log_file_path, DRIVE_DIR + log_file_path)
        print("💾 Model checkpoint saved successfully in folder: ", vm_checkpoint_folder, "and copied to: ", drive_checkpoint_folder)

In [ ]:
# @title Visualize evaluation scores during training { display-mode: "form" }
# This cell extracts logged evaluation scores and visualizes them
# You may still run this cell even if your training execution was interrupted,
# provided that the log file contains some evaluation results
which_training_round = "20260618_2217__TRAIN"
root_dir = DRIVE_DIR # either PROJECT_DIR or DRIVE_DIR
result_folder_path = f'{root_dir}/result/{which_training_round}'
# 1. Extract the evaluation histrory from the log file
exec_command_str = Extract_from_LogFile(result_folder_path, 'eval_result')
# print(exec_command_str)
exec(exec_command_str)

# 2. Plot eval results and save it as a png file
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(eval_result) + 1), eval_result)
plt.grid(True)
plt.title('Evaluation Results Over Training Epochs')
plt.xlabel('Epoch')
plt.ylabel('Average Tour Cost')
plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True)) # force integer x-axis ticks
plt.savefig(f'{result_folder_path}/eval_result.png')
print(f"\n📈 Evaluation results plotted and saved to: {result_folder_path}/eval_result.png")


For a successful training round, you would see in the final plot that the evaluation loss have converged after sufficiently many epochs. If you still see a trend of decay instead of the loss curve plateauing, it means that the model still has potential to improve. In that case, increase the number of epochs and/or the training dataset size. If the eval score does not go down, consider increasing your model capacity and/or reduce the variation in your training data.

# 🎯 Part 6: Inference & Benchmarking - Your Task

## 6.1 POMO Inference

After you have trained a model with a good evaluation score, it is time to put it to use. By "inference" we mean the process of using the trained agent to make real-time decisions in the environment. Recall that the model outputs a probability distribution over the next actions at any given state. We will use the so-called *greedy* rollout strategy, i.e. the actor always selects the action with the highest probability in the output distribution. Thus, given an input CVRP instance, the actor starts from the inital group state (empty tours) and iteratively selects the next action in a greedy manner until a terminal state is reached. We then simply take the tour within the group state with the lowest cost as our final solution. Notice how group states force the model to start generating solutions from each individual customer node, so this method can only perform better than decoding one single tour in which the actor chooses the first customer to visit.


You can see a mock inference process in action in the cell below, or open `visualizations/pomo_inference.html` in your browser:

In [ ]:
# @title Visualize POMO inference process { display-mode: "form" }
html = pathlib.Path("visualizations/pomo_inference.html").read_text(encoding="utf-8")
b64 = base64.b64encode(html.encode()).decode()
HTML(f'<iframe src="data:text/html;base64,{b64}" '
     f'width="100%" height="920" style="border:1px solid #ddd;border-radius:8px"></iframe>')

The pseudocode of the POMO training procedure is given in the cell below. You may find it useful for implementation.

<div style="border: 10px solid #ddd; margin: 0 auto">
    <div style="padding: 30px">
        <span style="color: red; font-weight: bold; font-style: italic; font-size: 20px;">
        POMO INFERENCE
        </span>
        
***INPUT*** An actor $\pi_\theta$, a CVRP instance $\mathcal{G}$ of size $n$
- ***Step 1:*** Apply greedy decoding on $\mathcal{G}$ using $\pi_\theta$ to get a grouped solution tour $\tau=(\tau_1,\ldots,\tau_n)$
- ***Step 2:*** Find the best solution $k^*:=\arg\max_{k=1,\ldots n} \mathcal{R}(\tau_k)$ with the highest return (i.e. lowest cost)

***OUTPUT:*** The best solution $\tau_{k^*}$</div></div>



## 🎯 6.2 POMO Inference Implementation: Your Task

Below you will implement the inference process of POMO, which is very similar to the training process except that actions are greedy (i.e. maximizers of the output probability) instead of being randomly sampled.

In [ ]:
# @title Implement the solve_with_POMO_actor function { display-mode: "form" }
def solve_with_POMO_actor(grouped_actor, demands, cost_matrix):
    """
    Solves a CVRP instance using the trained POMO actor by performing a rollout of the greedy policy.
    Parameters:
    grouped_actor: the trained ACTOR instance to use for solving
    demands: LongTensor of shape (batch, problem+1, 1)
    cost_matrix: Tensor of shape (batch, problem+1, problem+1) containing the pairwise travel costs
    Returns:
    best_cost: the cost of the best solution found (negative reward)
    best_sol: Tensor of shape (batch, selected_count) containing the sequence of visited nodes in the
              best solution found, where selected_count is the number of nodes visited in the solution
    """

    device = grouped_actor.device
    batch_s = demands.size(0)
    group_s = demands.size(1)-1  # = problem size
    with torch.no_grad():
        # 0. Move input tensors to the correct device
        demands = demands.to(device)
        cost_matrix = cost_matrix.to(device)
        features = cvrp.compute_batched_features(generator, cost_matrix, demands)
        
        # 1. Initialize Environment and Grouped Actor
        env = GROUP_ENVIRONMENT(demands, features, cost_matrix)
        group_state, reward, done = env.reset(group_size=group_s)
        grouped_actor.reset(group_state, env)

        # 2. Make the first move (which is fixed)
        # 🎯 TODO: create a LongTensor for the first action on device
        first_action = ...
        
        # 🎯 TODO: make a step in env based on first_action
        group_state, reward, done = ...

        # 3. Make the second move (which is fixed)
        second_action = torch.arange(1, group_s+1, dtype=torch.long, device=device)[None, :].expand(batch_s, group_s)
        # 🎯 TODO: make a step in env based on second_action
        group_state, reward, done = ...
        

        # 4. Subsequent Moves: follow the greedy policy until done
        while not done:
            # 4a. Get action probabilities from the grouped actor based on current group_state
            action_probs = grouped_actor.get_action_probabilities(group_state)
            # shape = (batch, group, problem+1)

            # 4b. Select the greedy action (argmax) for each tour
            # 🎯 TODO: select actions with higest probability for each tour
            # Hint: Use tensor.argmax(dim=...)
            action = ... # shape = (batch, group)

            # 5. Force finished tours to stay at depot
            action[group_state.finished] = 0

            # 6. Make a step in the environment based on the selected actions
            # 🎯 TODO: make a step in env based on action
            group_state, reward, done = ...

        # 7. After done, get the best solution and its cost from the group state and reward
        max_reward, argmax_group_idx = reward.max(dim=1) # best rollouts; shape = (batch,), (batch,)
        best_sols = group_state.selected_node_list[torch.arange(batch_s), argmax_group_idx, :] # shape = (batch, selected_count)

    return -max_reward.to('cpu')[0], best_sols.to('cpu')[0]

## 🎯 6.3 Load Model (Checkpoint) for Evaluation - Your Task

In [ ]:
# 🎯 TODO: Replace these folder names by the model/checkpoint you want to evaluate
which_training_round, which_checkpoint = "20260618_2217__TRAIN", "CheckPoint_ep00060"
root_dir = DRIVE_DIR # PROJECT_DIR or DRIVE_DIR
trained_actor = load_from_checkpoint(which_training_round, 
                                     which_checkpoint, root_dir=root_dir)
trained_actor.eval() # set to eval mode

## 6.4 Visualize the Solution from Your RL Actor

In [ ]:
p_size, seed = 60, 99
EvaluateModule.compare_POMO_with_baseline(solve_with_POMO_actor, trained_actor,
                    p_size=p_size, seed=seed, G_utm=G_utm, generator=generator)

## 6.5 Extensive Evaluation of Actor Performance
We are finally ready to benchmark our trained actor against the baseline solver!
- If you successfully trained your model, it should outperform the baseline in solution distance and time across (almost) all problem sizes.
- RL model can solve problems in batches, thus outperforming the baseline in terms of solution time

In [ ]:
# @title Extensive Evaluation Against OR-Tools Baseline { display-mode: "form" }
n_instances_per_size = 200
eval_seed = 999 # use the same seed for different methods to ensure fair comparison
or_costs, or_times, actor_costs, actor_times = EvaluateModule.evaluate_both_solvers(
    actor = trained_actor,
    n_instances_per_size=n_instances_per_size,
    p_sizes=P_SIZES,
    generator=generator,
    seed=eval_seed
) # shape for costs: (len(P_SIZES), n_instances_per_size), shape for times: (len(P_SIZES),)

In [ ]:
# @title Visualize Evaluation Results { display-mode: "form" }
# Plot the extensive evaluation results and save as png file
fig = visualize_solver_performance(cost_baseline=or_costs, time_baseline=or_times,
                                   cost_actor=actor_costs, time_actor=actor_times,
                                   n_instances_per_size=n_instances_per_size, eval_seed=eval_seed,
                                   figsize=(16,10))
# Run the following if you want to save this plot
extensive_evaluation_save_path = f'{root_dir}/result/{which_training_round}/{which_checkpoint}/extensive_evaluation.png'
fig.savefig(extensive_evaluation_save_path)
print('📈 Extensive evaluation results plotted and saved to: ', extensive_evaluation_save_path)

***Questions***
1. What can you conclude from your evaluation results?
2. Where is your model underperforming and how can you improve it?

# 🎯 Part 7: Next Steps


Congratulations 🎉👏🎊!
You have now finished training an RL model which performs vehicle routing!


## 🎯 7.1 After Finishing the First Working Version


Here are some next steps to improve your model performance:
1. **Hyperparameter tuning**: Try different parameters for model size, dataset size and learning rate parameters. Bigger models are more expressive (and hence have more potential) but takes more data to train. Batch size also makes a difference: our dataloader produces batches of different input sizes. This means larger batch size will result in more drastic updates (assuming the number of batches is fixed) and the training will be more unstable, but if you keep the batch sizes tool small there would not be enough information for the model to learn efficiently, i.e. you might need significantly more batches and epochs.
2. **Curriculum learning**: Experiement with the distribution of input instance size during training. The distribution should be stable in the beginning for the model to "quickly learn the basics", and in later epochs you can increase the difficulty by increasing the variance of the distribution and/or change the value of the mean. A larger mean is harder for the model because on average there are more customers to deliver to, while a larger variance is harder because the model would be forced to do well on problems of very different sizes.



## 7.2 Reflections

Here are some questions to think about; they may or may not appear in your project defense :)
1. Do you understand at a high level what policy gradient algorithms are? How do they relate to the RL algorithms introduced in class (e.g. SARSA)?
2. Can you describe how POMO models CVRP into an RL setting? What are the states, actions, environments, transitions and rewards?
3. Can you describe the training & inference algorithm for POMO?
4. Can you explain how different hyperparameters (batch size, number of samples, architecture parameters, learning rates) can affect the training process of the actor?
5. Beyond what was mentioned in 7.1, can you think of other possible ways to improve the performance of RL models? How would do things differently?
6. There are many variants of CVRP formulations. For example, we might assume that a large number of vehicles of a fixed capacity is available regardless of how many customers we serve, but using each vehicle incurs a fixed cost. You would like to minimize the total cost while serving all customers subject to the capacity constraint, where the cost is a combination of the total travel distance and the fixed costs of using vehicles. How would you modify the RL formulation to train an actor for this problem?
7. If you were to design an RL solver for CVRP from scratch, how would you do it instead of POMO? Can you adapt any RL algorithm we learnt in class to solve this?
8. As your business grow, you may have multiple depots. How do you factor this into your RL model?